In [ ]:
########TASK1############
import random
input_text=open('/content/sample_data/input.txt','r')
grid_length=int(input_text.readline())
grid_height=int(input_text.readline())
population_num=int(input_text.readline())
block_info=eval(input_text.readline())
interconnection=eval(input_text.readline())
alpha = int(input_text.readline())
beta = int(input_text.readline())
gamma =int(input_text.readline())


def population_create(population_num,grid_length,grid_height,block_info):
 population=[]
 for j in range(population_num):
  chromosome=[]
  for i in range(len(block_info)):
   x=random.randint(0,grid_length-block_info[i][1])
   y=random.randint(0,grid_height-block_info[i][2])
   chromosome.append((x,y))
  population.append(chromosome)
 return population


####fitness calculation shuru###


def wiring_distances__claculate(block_info,population,interconnection):
##midpoint calculate start

 population_midpoint=[]
 idx=0
 for i in population:
   each_block_midpoint=[]

   for j in block_info:
    bottom_left_x=i[idx][0]
    bottom_left_y=i[idx][1]

    bottom_right_x=bottom_left_x+ j[1]
    bottom_right_y=bottom_left_y

    top_left_x=bottom_left_x
    top_left_y=bottom_left_y+j[2]

    center_x = (bottom_right_x + top_left_x)/2
    center_y = (bottom_right_y + top_left_y)/2
    each_block_midpoint.append((center_x,center_y))
    idx+=1
   population_midpoint.append(each_block_midpoint)
   idx=0


##calculate wiring distances
 total_dis=0
 dic={}
 a=0
 for m in block_info:
  dic[m[0]]=a   ##each block pos onujayi index assign
  a+=1

 each_chromosome_total_wiring_distance=[]
 dis=0
 for i in population_midpoint:
  for n in interconnection:
   index1=dic[n[0]] ##each block er index dibe like alu=0
   index2=dic[n[1]]
   x1=i[index1][0]
   y1=i[index1][1]
   x2=i[index2][0]
   y2=i[index2][1]
   dis+=((x1-x2)**2+(y1-y2)**2)**0.5
  each_chromosome_total_wiring_distance.append(dis)
  dis=0
 return each_chromosome_total_wiring_distance


def bounding_area_calculate(block_info,population):
 bounding_area=[]
 for x in population:
  x_min=0
  y_min=0
  x_max=0
  y_max=0
  initial=0
  area=0
  for y in block_info:
    if x[initial][0]<x_min:
      x_min=x[initial][0]

    if x[initial][1]<y_min:
      y_min=x[initial][1]

    if x[initial][0]+y[1]>x_max:
      x_max=x[initial][0]+y[1]

    if x[initial][1]+y[2]>y_max:
      y_max=x[initial][1]+y[2]
    initial+=1
  area= (x_max - x_min) * (y_max - y_min)
  bounding_area.append(area)
 return bounding_area



def overlap_calculate(block_info,population  ):
 overlap_count_each_chromosome=[]
 for c in population:
   overlap_count=0
   for u in range(len(block_info)):
     for v in range(u+1,len(block_info)):
       A_left = c[u][0]
       A_right =c[u][0]+ block_info[u][1]
       A_bottom =c[u][1]
       A_top = c[u][1]+ block_info[u][2]

       B_left = c[v][0]
       B_right =c[v][0]+ block_info[v][1]
       B_bottom =c[v][1]
       B_top = c[v][1]+ block_info[v][2]

       if not (A_right <= B_left or A_left >= B_right or A_bottom >= B_top or A_top <= B_bottom):
          overlap_count+=1
   overlap_count_each_chromosome.append(overlap_count)
 return overlap_count_each_chromosome


def fitness_calculate(block_info,population,interconnection):
   a=wiring_distances__claculate(block_info,population,interconnection)
   b=bounding_area_calculate(block_info,population)
   c=overlap_calculate(block_info,population)
   fitness_store= []
   for i in range(len(population)):
        total_fitness = -(alpha * c[i] + beta * a[i] + gamma * b[i])
        total_fitness=round(total_fitness,2)
        fitness_store.append(total_fitness)
   return fitness_store, a, b, c

def Parent_selection(population):
   idx_parent1 = random.randint(0, len(population)-1)
   while True:
    idx_parent2 = random.randint(0, len(population)-1)
    if idx_parent2 != idx_parent1:
        break
   return population[idx_parent1],population[idx_parent2]

def single_point_crossover(parent1, parent2):
    random_point = random.randint(1, len(parent1) - 1)
    child1 = parent1[:random_point] + parent2[random_point:]
    child2 = parent2[:random_point] + parent1[random_point:]
    return child1, child2

def mutation(child, grid_length, grid_height, block_info):
  mutation_rate = 10/100
  if random.random() < mutation_rate:
        random_idx = random.randint(0, len(child)-1)
        block_name, width, height = block_info[random_idx]
        x = random.randint(0, grid_length-width)
        y = random.randint(0, grid_height - height)
        child[random_idx] = (x, y)
  return child

def genetic_algo(population_num, grid_length, grid_height, block_info, interconnection):
  max_gen=15
  best_total_fitness = float('-inf')
  total_wiring_length=0
  total_bounding_box_area=0
  total_overlaps_counts=0
  best_chromosome=None
  population = population_create(population_num,grid_length,grid_height,block_info)
  for k in range(max_gen):
    fitness_scores,wiring_distances,bounding_areas,overlap_counts=fitness_calculate(block_info,population,interconnection)
    best_idx = 0
    best_score = float('-inf')
    for i in range(len(population)):
            if overlap_counts[i] == 0:
                score = -(beta * wiring_distances[i] + gamma * bounding_areas[i])
                if score > best_score:
                    best_score = score
                    best_idx = i
            else:

                if fitness_scores[i] > best_score:
                    best_score = fitness_scores[i]
                    best_idx = i


    if best_score>best_total_fitness:
      best_total_fitness=best_score
      total_wiring_length=wiring_distances[best_idx]
      total_bounding_box_area=bounding_areas[best_idx]
      total_overlaps_counts=overlap_counts[best_idx]
      best_chromosome=population[best_idx]

    new_population = [best_chromosome]
    while len(new_population) < population_num:
            parent1, parent2 = Parent_selection(population)
            child1, child2 = single_point_crossover(parent1,parent2)
            child1 = mutation(child1, grid_length,grid_height,block_info)
            child2 = mutation(child2, grid_length,grid_height,block_info)
            new_population.extend([child1,child2])
    population = new_population[:population_num]
    if total_overlaps_counts==0 and k>6:
      break
  return best_chromosome,best_total_fitness,total_wiring_length,total_bounding_box_area,total_overlaps_counts


best_chromosome,best_total_fitness,total_wiring_length,total_bounding_box_area,total_overlaps_counts=genetic_algo(population_num, grid_length,grid_height,block_info,interconnection)
print('######TASK1######')

print('Best total fitness value',best_total_fitness)
print('Total wiring length',total_wiring_length)
print('The total bounding box area',total_bounding_box_area)
print('the total overlap counts',total_overlaps_counts)
print('Best chromosome representation',best_chromosome)




#####################TASK2###############



print('#####TASK 2########')

def two_point_crossover(parent1, parent2):
    point1 = random.randint(0, len(parent1)-3)
    point2 = random.randint(point1+1,len(parent1) - 1)

    child1 = parent1[:point1] + parent2[point1:point2+1] + parent1[point2+1:]
    child2 = parent2[:point1] + parent1[point1:point2+1] + parent2[point2+1:]

    return child1, child2
parent1,parent2=Parent_selection(population)
child1,child2=two_point_crossover(parent1,parent2)
print(f'Child1: {child1}, Child2: {child2}')






######TASK1######
Best total fitness value -431.9978590276929
Total wiring length 55.998929513846456
The total bounding box area 320
the total overlap counts 0
Best chromosome representation [(0, 1), (0, 12), (7, 10), (10, 1), (14, 12), (15, 7)]
#####TASK 2########


NameError: name 'population' is not defined